# Eye tracking · 1.4 · Remove task-evoked components


In [1]:
from pathlib import Path
import sys
base_path = next(
    p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "CODE").is_dir() and (p / "DATA").is_dir()
)
sys.path.insert(0, str(Path(base_path) / 'CODE'))


In [2]:
import numpy as np
import statsmodels.api as sm
import pickle
import os
import scipy.io as sio
import matplotlib.pyplot as plt
from scipy import signal

In [5]:
# parameters
base_path = base_path
pupil_size_data_path = f"{base_path}/DATA/pupillometry/pupil_size/pupil_size_preprocessed.pkl"
microsaccades_data_path = f"{base_path}/DATA/pupillometry/microsaccades/microsaccades.pkl"

BASE_REG_PATH = f"{base_path}/DATA/pupillometry/regression_events"
output_path = f"{base_path}/DATA/pupillometry/pupil_residuals"

task_names = ['0nback', '0nback', '1nback', '1nback', '2nback', '2nback']

# decimation to match fMRI TR
decimate_b = True
decimate_factor = 1000

# files to exclude (subject_id, task_id) after visual inspection
to_delete = [(1, 3), (1, 4), (1, 5), (1, 6), (4, 4), (27, 1), (27, 2),
              (27, 3), (27, 4), (27, 5), (27, 6), (28, 1), (30, 1), (32, 2)]


In [6]:
# functions for loading and processing event files
def load_events(path):
    """Load event information from a text file."""
    x = np.loadtxt(path)
    if x.ndim == 1:
        x = x[None, :]
    return x  #onset, duration, weight

def events_to_regressor(events, n_tp, dt):
    """Convert event information to a time series regressor."""
    t = np.arange(n_tp) * dt
    r = np.zeros(n_tp)

    for onset, duration, weight in events:
        mask = (t >= onset) & (t < onset + duration)
        r[mask] += weight

    return r

## pupil size

In [7]:
# load data and create output directory

with open(pupil_size_data_path, 'rb') as f:
    data = pickle.load(f)

if decimate_b:
    os.makedirs(f"{output_path}/pupil_size/decimated_{int(decimate_factor/500)}s", exist_ok=True)
else:
    os.makedirs(f"{output_path}/pupil_size/original_0.02s", exist_ok=True)

for i in range(len(data['preprocessed_timeseries'])):
    sub_id = data['subject'][i]
    task_id = data['task_ids'][i]
    if decimate_b:
        path = f'{output_path}/pupil_size/decimated_{int(decimate_factor/500)}s/sub{sub_id:02d}_{task_id:02d}.mat'
    else:
        path = f'{output_path}/pupil_size/original_0.02s/sub{sub_id:02d}_{task_id:02d}.mat'

    if (sub_id, task_id) in to_delete:
        print(f'Skipping sub{sub_id:02d}_{task_id:02d} as it is marked for deletion.')
        continue

    series = data['preprocessed_timeseries'][i]
    if decimate_b:
        series = signal.decimate(series, decimate_factor, ftype='fir', zero_phase=True)

    y = series
    n_tp = len(y)
    dt = int(decimate_factor/500) if decimate_b else 1/500   

    r1 = events_to_regressor(load_events(f"{BASE_REG_PATH}/sub{sub_id:02d}/sub{sub_id:02d}/{task_names[task_id - 1]}/block.txt"), n_tp, dt)
    r2 = events_to_regressor(load_events(f"{BASE_REG_PATH}/sub{sub_id:02d}/sub{sub_id:02d}/{task_names[task_id - 1]}/targets.txt"), n_tp, dt)
    r3 = events_to_regressor(load_events(f"{BASE_REG_PATH}/sub{sub_id:02d}/sub{sub_id:02d}/{task_names[task_id - 1]}/trials.txt"), n_tp, dt)

    X = np.column_stack([r1, r2, r3])
    X = sm.add_constant(X)   

    model = sm.OLS(y, X)
    results = model.fit()
    residuals = results.resid

    start = int(np.ceil(500/decimate_factor)) if decimate_b else 500

    sio.savemat(path, {'pupil_timeseries': residuals[start:]})
    print(f'Saved sub{sub_id:02d}_{task_id:02d}.mat')

Skipping sub01_03 as it is marked for deletion.
Skipping sub01_04 as it is marked for deletion.
Skipping sub01_05 as it is marked for deletion.
Skipping sub01_06 as it is marked for deletion.
Saved sub02_01.mat
Saved sub02_02.mat
Saved sub02_03.mat
Saved sub02_04.mat
Saved sub02_05.mat
Saved sub02_06.mat
Saved sub03_01.mat
Saved sub03_02.mat
Saved sub03_03.mat
Saved sub03_04.mat
Saved sub03_05.mat
Saved sub03_06.mat
Saved sub04_01.mat
Saved sub04_02.mat
Saved sub04_03.mat
Skipping sub04_04 as it is marked for deletion.
Saved sub05_01.mat
Saved sub05_02.mat
Saved sub05_03.mat
Saved sub05_04.mat
Saved sub05_05.mat
Saved sub05_06.mat
Saved sub06_01.mat
Saved sub06_02.mat
Saved sub06_03.mat
Saved sub06_04.mat
Saved sub06_05.mat
Saved sub06_06.mat
Saved sub07_01.mat
Saved sub07_02.mat
Saved sub07_03.mat
Saved sub07_04.mat
Saved sub07_05.mat
Saved sub07_06.mat
Saved sub08_01.mat
Saved sub08_02.mat
Saved sub08_03.mat
Saved sub08_04.mat
Saved sub08_05.mat
Saved sub08_06.mat
Saved sub09_01.mat


## Microsaccades

In [8]:
with open(microsaccades_data_path, 'rb') as f:
    data = pickle.load(f)

data = data.copy()

all_microsaccades = np.full((36, 6, 494), np.nan)
all_residuals = np.full((36, 6, 494), np.nan)

# remove irrealistic microsaccades (longer than 1 second)
for idx in data.index:
    starts = np.asarray(data.at[idx, "start_times"])
    ends   = np.asarray(data.at[idx, "end_times"])

    mask = (ends - starts) < 1
    data.at[idx, "start_times"] = starts[mask]
    data.at[idx, "end_times"]   = ends[mask]

# Load and preprocess pupillometry data
bins = np.arange(1, 494, 1)  # 1 second

if decimate_b:
    os.makedirs(f"{output_path}/microsaccades/decimated_{int(decimate_factor/500)}s", exist_ok=True)
else:
    os.makedirs(f"{output_path}/microsaccades/original_1s", exist_ok=True)

for i in range(len(data["start_times"])):
    sub_id = data["subject"].iloc[i]
    task_id = data["task_ids"].iloc[i]

    if (sub_id, task_id) in to_delete:
        print(f'Skipping sub{sub_id:02d}_task{task_id:02d} as it is marked for deletion.')
        continue

    a = np.histogram(data["start_times"].iloc[i], bins)
    if decimate_b:
        path = f'{output_path}/microsaccades/decimated_{int(decimate_factor/500)}s/sub{sub_id:02d}_{task_id:02d}.mat'
    else:
        path = f'{output_path}/microsaccades/original_1s/sub{sub_id:02d}_{task_id:02d}.mat'

    if sum(a[0]) == 0:
        print(f'No microsaccades for sub{sub_id:02d}_task{task_id:02d}, skipping save.')
        continue

    if decimate_b:
        new_decimate_factor = int(decimate_factor / 500)
        a = signal.decimate(a[0], new_decimate_factor, ftype='fir', zero_phase=True)
    else:
        a = a[0]
    
    all_microsaccades[sub_id - 1, task_id - 1, :len(a)] = a

    y = a
    n_tp = len(y)
    dt = int(new_decimate_factor) if decimate_b else 1

    r1 = events_to_regressor(load_events(f"{BASE_REG_PATH}/sub{sub_id:02d}/sub{sub_id:02d}/{task_names[task_id - 1]}/block.txt"), n_tp, dt)
    r2 = events_to_regressor(load_events(f"{BASE_REG_PATH}/sub{sub_id:02d}/sub{sub_id:02d}/{task_names[task_id - 1]}/targets.txt"), n_tp, dt)
    r3 = events_to_regressor(load_events(f"{BASE_REG_PATH}/sub{sub_id:02d}/sub{sub_id:02d}/{task_names[task_id - 1]}/trials.txt"), n_tp, dt)

    X = np.column_stack([r1, r2, r3])
    X = sm.add_constant(X)

    model = sm.OLS(y, X)
    results = model.fit()
    residuals = results.resid

    all_residuals[sub_id - 1, task_id - 1, :len(residuals)] = residuals

    sio.savemat(path, {'timeseries': residuals})
    print(f'Saved sub{sub_id:02d}_task{task_id:02d}.mat')

No microsaccades for sub01_task01, skipping save.
No microsaccades for sub01_task02, skipping save.
Skipping sub01_task03 as it is marked for deletion.
Skipping sub01_task04 as it is marked for deletion.
Skipping sub01_task05 as it is marked for deletion.
Skipping sub01_task06 as it is marked for deletion.
Saved sub02_task01.mat
Saved sub02_task02.mat
Saved sub02_task03.mat
Saved sub02_task04.mat
Saved sub02_task05.mat
Saved sub02_task06.mat
Saved sub03_task01.mat
Saved sub03_task02.mat
Saved sub03_task03.mat
Saved sub03_task04.mat
Saved sub03_task05.mat
Saved sub03_task06.mat
Saved sub04_task01.mat
Saved sub04_task02.mat
Saved sub04_task03.mat
Skipping sub04_task04 as it is marked for deletion.
Saved sub05_task01.mat
Saved sub05_task02.mat
Saved sub05_task03.mat
Saved sub05_task04.mat
Saved sub05_task05.mat
Saved sub05_task06.mat
Saved sub06_task01.mat
Saved sub06_task02.mat
Saved sub06_task03.mat
Saved sub06_task04.mat
Saved sub06_task05.mat
Saved sub06_task06.mat
Saved sub07_task01.